# Temporal Alignment — News and Liquidity Data

Loads cleaned GDELT news articles from the database, assigns each to its next available 
trading hour, merges with WTI hourly price and liquidity data, and saves the result 
back to the `liquidity` table in the database.

**Key design decisions:**
- `dt.ceil('h')` instead of `dt.round('h')` — always rounds forward to preserve causality
- Off-hours articles are forward-assigned to the next market open instead of discarded
- Articles published before the price dataset starts are discarded
- Zero-volume hours are discarded (market open artifacts)
- `assignment_gap` records hours between publication and assigned trading hour

### General imports

In [2]:
import pandas as pd
import numpy as np
import sqlite3

### Connect to database

In [3]:
conn = sqlite3.connect("../01_data/wti_thesis.db")
print("Connected to wti_thesis.db")

Connected to wti_thesis.db


### Load hourly price data from market_context

In [4]:
df_price = pd.read_sql("""
    SELECT datetime_hour, log_volume, price_range, log_return, amihud
    FROM market_context
""", conn)

df_price['datetime_hour'] = pd.to_datetime(df_price['datetime_hour'], utc=True)
df_price = df_price.set_index('datetime_hour').sort_index()

print(f"Price records: {len(df_price)}")
print(f"Range: {df_price.index.min()} to {df_price.index.max()}")

Price records: 11193
Range: 2024-05-10 13:00:00+00:00 to 2026-05-08 20:00:00+00:00


### Load news articles from database

For each article, the publication timestamp is ceiled to the next available trading hour.
Off-hours articles (nights, weekends) are forward-assigned to the next market open rather 
than discarded — news published when the market is closed still influences trader behavior 
when it reopens. Only articles published before the price dataset starts are discarded.

In [5]:
df_news = pd.read_sql("""
    SELECT id, title, datetime, domain, url, body_valid
    FROM articles
""", conn)

df_news['datetime'] = pd.to_datetime(df_news['datetime'], utc=True)
price_index = df_price.index.sort_values()

print(f"Articles loaded: {len(df_news)}")
print(f"Price range: {price_index.min()} to {price_index.max()}")

Articles loaded: 16326
Price range: 2024-05-10 13:00:00+00:00 to 2026-05-08 20:00:00+00:00


### Assign each article to next available trading hour

In [6]:
def get_next_trading_hour(timestamp, price_index):
    future_hours = price_index[price_index >= timestamp]
    if len(future_hours) > 0:
        return future_hours[0]
    return None

print("Assigning articles to next available trading hour...")
df_news['datetime_hour'] = df_news['datetime'].apply(
    lambda ts: get_next_trading_hour(ts, price_index)
)

# Compute assignment gap in hours
df_news['assignment_gap'] = (
    df_news['datetime_hour'] - df_news['datetime']
).dt.total_seconds() / 3600

# Discard articles beyond price dataset range
before = len(df_news)
df_news = df_news.dropna(subset=['datetime_hour']).reset_index(drop=True)

print(f"Articles before: {before}")
print(f"Discarded (beyond price range): {before - len(df_news)}")
print(f"Articles after: {len(df_news)}")
print(f"\nAssignment gap distribution:")
print(f"< 1h:   {(df_news['assignment_gap'] < 1).sum()}")
print(f"1-2h:   {((df_news['assignment_gap'] >= 1) & (df_news['assignment_gap'] < 2)).sum()}")
print(f"2-24h:  {((df_news['assignment_gap'] >= 2) & (df_news['assignment_gap'] < 24)).sum()}")
print(f"> 24h:  {(df_news['assignment_gap'] >= 24).sum()}")

Assigning articles to next available trading hour...
Articles before: 16326
Discarded (beyond price range): 0
Articles after: 16326

Assignment gap distribution:
< 1h:   10166
1-2h:   431
2-24h:  1101
> 24h:  4628


### Merge articles with liquidity data

Left join on datetime_hour — each article receives the liquidity metrics of its 
assigned trading hour. Articles whose assigned hour has no price data (zero volume 
market open artifacts, or hours before price dataset starts) are discarded.

In [7]:
df_price_reset = df_price.reset_index()

df_aligned = df_news.merge(
    df_price_reset[['datetime_hour', 'log_volume', 'price_range', 'log_return', 'amihud']],
    on='datetime_hour',
    how='left'
)

print(f"Articles after merge: {len(df_aligned)}")
print(f"Without price data: {df_aligned['log_volume'].isna().sum()}")

Articles after merge: 16326
Without price data: 0


### Discard articles without valid price data

Removes articles published before the price dataset starts and zero-volume hours 
(market open artifacts where no trades were recorded).

In [8]:
price_start = df_price.index.min()

df_final = df_aligned[
    (df_aligned['datetime'] >= price_start) &
    (df_aligned['log_volume'].notna()) &
    (df_aligned['log_volume'] > 0)
].reset_index(drop=True)

print(f"Articles with valid price data: {len(df_final)}")
print(f"Temporal range: {df_final['datetime'].min()} to {df_final['datetime'].max()}")
print(f"\nLiquidity statistics:")
print(df_final[['log_volume', 'price_range', 'amihud']].describe().round(4))
print(f"\nContemporaneous (gap < 2h): {(df_final['assignment_gap'] < 2).sum()}")
print(f"Off-hours (gap >= 2h): {(df_final['assignment_gap'] >= 2).sum()}")

Articles with valid price data: 12024
Temporal range: 2024-05-10 13:15:00+00:00 to 2026-03-01 00:30:00+00:00

Liquidity statistics:
       log_volume  price_range      amihud
count  12024.0000   12024.0000  12024.0000
mean       8.4930       0.3921      0.0000
std        1.5407       0.3262      0.0000
min        0.6931       0.0000      0.0000
25%        7.4868       0.2000      0.0000
50%        8.6627       0.3200      0.0000
75%        9.6572       0.5000      0.0000
max       12.1323       5.8700      0.0021

Contemporaneous (gap < 2h): 9924
Off-hours (gap >= 2h): 2100


### Save to database

In [9]:
df_liquidity = df_final[['id', 'datetime_hour', 'log_volume', 
                          'price_range', 'log_return', 'amihud', 
                          'assignment_gap']].copy()

df_liquidity.columns = ['article_id', 'datetime_hour', 'log_volume',
                         'price_range', 'log_return', 'amihud', 'assignment_gap']
df_liquidity['datetime_hour'] = df_liquidity['datetime_hour'].astype(str)

df_liquidity.to_sql('liquidity', conn, if_exists='replace', index=False)
print(f"Liquidity records saved: {len(df_liquidity)}")

Liquidity records saved: 12024


### Verify

In [10]:
sample = pd.read_sql("""
    SELECT a.title, a.datetime, l.log_volume, l.price_range, l.assignment_gap
    FROM articles a
    JOIN liquidity l ON a.id = l.article_id
    LIMIT 5
""", conn)
print(sample)

counts = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN assignment_gap < 2 THEN 1 ELSE 0 END) as contemporaneous,
        SUM(CASE WHEN assignment_gap >= 2 THEN 1 ELSE 0 END) as off_hours
    FROM liquidity
""", conn)
print(counts)

conn.close()
print("\nAlignment complete.")

                                               title  \
0  Stock market today : Global shares trade highe...   
1  Goldman Sachs sees crude price rangebound with...   
2  Global shares trade higher after Wall Street r...   
3  US Futures , Global Markets Storm Higher , Eye...   
4  Oil price news : Oil extends 2 - day climb on ...   

                    datetime  log_volume  price_range  assignment_gap  
0  2024-05-10 13:15:00+00:00   10.176221     0.519997            0.75  
1  2024-05-10 13:15:00+00:00   10.176221     0.519997            0.75  
2  2024-05-10 13:15:00+00:00   10.176221     0.519997            0.75  
3  2024-05-10 13:30:00+00:00   10.176221     0.519997            0.50  
4  2024-05-10 13:30:00+00:00   10.176221     0.519997            0.50  
   total  contemporaneous  off_hours
0  12024             9924       2100

Alignment complete.
